### Q1 NLP Preprocessing Pipeline

In [3]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download necessary NLTK data including the required punkt_tab
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
def preprocess_sentence(sentence):
    # 1. Tokenize the sentence
    tokens = word_tokenize(sentence)
    print("1. Original Tokens:", tokens)

    # 2. Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens_no_stopwords = [word for word in tokens if word.lower() not in stop_words]
    print("2. Tokens Without Stopwords:", tokens_no_stopwords)

    # 3. Apply Stemming
    stemmer = PorterStemmer()
    stemmed_words = [stemmer.stem(word) for word in tokens_no_stopwords]
    print("3. Stemmed Words:", stemmed_words)

    return tokens, tokens_no_stopwords, stemmed_words

# Test sentence
sentence = "NLP techniques are used in virtual assistants like Alexa and Siri."
preprocess_sentence(sentence)

1. Original Tokens: ['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']
2. Tokens Without Stopwords: ['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri', '.']
3. Stemmed Words: ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri', '.']


(['NLP',
  'techniques',
  'are',
  'used',
  'in',
  'virtual',
  'assistants',
  'like',
  'Alexa',
  'and',
  'Siri',
  '.'],
 ['NLP',
  'techniques',
  'used',
  'virtual',
  'assistants',
  'like',
  'Alexa',
  'Siri',
  '.'],
 ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri', '.'])

#### **Short Answer Questions**

**1. What is the difference between stemming and lemmatization? Provide examples with the word “running.”**
*   **Stemming:** A crude, rule-based heuristic process that chops off the ends of words to find their base or root form. It does not understand grammatical context or word meanings, which can result in non-dictionary words.
    *   *Example:* "running" -> **"run"**, "runs" -> **"run"**, "ran" -> **"ran"**.
*   **Lemmatization:** A detailed morphological analysis that matches words to their true base dictionary form (lemma) using vocabulary and part-of-speech context.
    *   *Example:* "running" (verb) -> **"run"**, "runs" -> **"run"**, "ran" -> **"run"**.

---

**2. Why might removing stop words be useful in some NLP tasks, and when might it actually be harmful?**
*   **When it is useful:** In **Information Retrieval**, **Document Classification**, and **Topic Modeling**, highly frequent words (such as "the", "is", "and") act as background noise. Removing them reduces the dimension of the data and directs algorithms to focus on content-carrying keywords.
*   **When it is harmful:** In **Sentiment Analysis**, dropping words like "not" (e.g., in "not good") completely reverses polarity. In **Machine Translation** and **Text Generation**, omitting stop words ruins sentence structure, syntax, and grammatical coherence.

### Q2 Named Entity Recognition with SpaCy

In [8]:
import spacy

# Load pipeline
nlp = spacy.load("en_core_web_sm")

# Given sentence
sentence = " LeBron James as the NBA player of the United States in Laker and won the NBA Final MVP in 2020."

# Process the sentence
doc = nlp(sentence)

# Print named entities in a clean, formatted table
print("Detected Named Entities:")
print(f"{'Entity':<20} | {'Label':<10} | {'Start':<6} | {'End':<6}")
print("-" * 50)
for ent in doc.ents:

    clean_text = ent.text.strip()
    print(f"{clean_text:<20} | {ent.label_:<10} | {ent.start_char:<6} | {ent.end_char:<6}")

Detected Named Entities:
Entity               | Label      | Start  | End   
--------------------------------------------------
LeBron James         | PERSON     | 1      | 13    
NBA                  | ORG        | 21     | 24    
the United States    | GPE        | 35     | 52    
Laker                | GPE        | 56     | 61    
MVP                  | ORG        | 84     | 87    
2020                 | DATE       | 91     | 95    


#### **Short Answer Questions**

**1. How does NER differ from POS tagging in NLP?**
*   **POS (Part-of-Speech) Tagging** assigns a grammatical category (such as noun, verb, adjective, or preposition) to *every single word* in a sentence based on its syntax and relationship with adjacent words.
*   **NER (Named Entity Recognition)** identifies and classifies only specific *real-world concepts or objects* (such as people, organizations, locations, dates, or quantities) that are represented by noun phrases or proper nouns. NER ignores ordinary verbs, adjectives, and non-entity nouns entirely.

---

**2. Describe two applications that use NER in the real world.**
*   **Information Extraction from Financial News:** Investment algorithms use NER to instantly scan thousands of news articles to extract company names, currencies, and financial values. This allows them to monitor market trends, track specific stocks, and execute automated sentiment-based trades.
*   **Search Engine Optimization and Query Processing:** Search engines use NER to parse and understand user intent. By recognizing specific locations (e.g., "restaurants in Kyoto") or entities (e.g., "movies starring Leonardo DiCaprio"), the search engine can serve targeted, structured results (like maps, biographies, or timetables) rather than simple keyword matches.

### Q3: Scaled Dot-Product Attention

In [6]:
import numpy as np

def softmax(x):
    # Apply softmax along the last axis
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    # Get the dimension of the key vector d
    d = Q.shape[-1]

    # 1. Compute dot product of Q and K transposed
    scores = np.dot(Q, K.T)

    # 2. Scale scores by the square root of d
    scaled_scores = scores / np.sqrt(d)

    # 3. Apply softmax to obtain the attention weights
    attention_weights = softmax(scaled_scores)

    # 4. Multiply by V to get the final output
    output = np.dot(attention_weights, V)

    return attention_weights, output

# Test inputs
Q = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
V = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])

# Run the attention mechanism
weights, output = scaled_dot_product_attention(Q, K, V)

print("1. Attention Weights Matrix (after softmax):")
print(weights)
print("\n2. Final Output Matrix:")
print(output)

1. Attention Weights Matrix (after softmax):
[[0.73105858 0.26894142]
 [0.26894142 0.73105858]]

2. Final Output Matrix:
[[2.07576569 3.07576569 4.07576569 5.07576569]
 [3.92423431 4.92423431 5.92423431 6.92423431]]


#### **Short Answer Questions**

**1. Why do we divide the attention score by √d in the scaled dot-product attention formula?**
*   As the dimension of the key vectors $d$ grows larger, the dot products also grow large in magnitude.
*   These extremely large values push the softmax function into regions with extremely small gradients (the vanishing gradient problem).
*   Scaling down by dividing by $\sqrt{d}$ helps keep the variance of the dot products around 1, ensuring stable gradients during backpropagation.

---

**2. How does self-attention help the model understand relationships between words in a sentence?**
*   Self-attention allows each word (represented as a query) to look at all other words in the sentence (represented as keys) and calculate a compatibility/relation score.
*   By applying these scores as weights over the values, the representation of a word at any layer becomes a weighted sum of all words in the sequence.
*   This enables the model to capture dynamic, contextual, and long-range dependencies regardless of their distance in the sentence (e.g., linking a pronoun "it" back to its antecedent noun "the cat" several words earlier).